# Netflix Business Insights Report

**Prepared by:** Inchara L
**Dataset:** Netflix titles (8,786 titles, 2008 to 25 September 2021)
**Tools:** Python, pandas, matplotlib, Plotly

## Executive Summary

Netflix's catalog in this dataset is dominated by Movies (69.7%), but TV Shows are gaining ground: their share of newly added titles rose from 25.0% in 2018 to 33.7% in 2021. Content comes mainly from a few markets: the United States supplies 38.1% of titles with a known country, and the top 10 countries supply 78.0%. The catalog is aimed mostly at teens and adults (75.7% of titles). The number of titles added each year peaked in 2019 (2,013) and levelled off afterwards, while the 2021 figure is incomplete because the data ends on 25 September 2021.

**Three recommendations**
1. Keep expanding TV series, which are gaining share.
2. Protect the strong base in the US, India and the UK, and test small investments in under-served countries.
3. Grow family and children's content, which is only 23.4% of the catalog.

## Data and Method

- **Source:** Provided as part of the internship task
- **Size:** 8,790 rows and 10 columns in the raw file; 8,786 rows after cleaning
- **Cleaning:** replaced hidden placeholders ("Not Given") with "Unknown", removed 4 duplicate titles, stripped extra spaces, and converted dates
- **Genres:** each title can carry several genres, so genre shares add up to more than 100%
- **Partial year:** 2021 is incomplete, so it is treated with caution

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

df = pd.read_csv("netflix_cleaned.csv", parse_dates=["date_added"])
df.shape

(8786, 16)

In [2]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8786 entries, 0 to 8785
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   show_id          8786 non-null   object        
 1   type             8786 non-null   object        
 2   title            8786 non-null   object        
 3   director         8786 non-null   object        
 4   country          8786 non-null   object        
 5   date_added       8786 non-null   datetime64[ns]
 6   release_year     8786 non-null   int64         
 7   rating           8786 non-null   object        
 8   duration         8786 non-null   object        
 9   listed_in        8786 non-null   object        
 10  year_added       8786 non-null   int64         
 11  month_added      8786 non-null   object        
 12  primary_country  8786 non-null   object        
 13  rating_group     8786 non-null   object        
 14  duration_value   8786 non-null   int64  

In [4]:
df[["release_year", "year_added", "duration_value"]].describe().round(1)

,release_year,year_added,duration_value
count,8786.0,8786.0,8786.0
mean,2014.2,2018.9,69.9
std,8.8,1.6,50.8
min,1925.0,2008.0,1.0
25%,2013.0,2018.0,2.0
50%,2017.0,2019.0,88.0
75%,2019.0,2020.0,106.0
max,2021.0,2021.0,312.0


In [5]:
df.nunique()

show_id            8786
type                  2
title              8786
director           4528
country              86
date_added         1713
release_year         74
rating               14
duration            220
listed_in           513
year_added           14
month_added          12
primary_country      86
rating_group          5
duration_value      210
duration_unit         2
dtype: int64

In [6]:
print("Empty cells:", df.isnull().sum().sum())
print("Unknown director:", (df["director"] == "Unknown").sum())
print("Unknown country:", (df["country"] == "Unknown").sum())

Empty cells: 0
Unknown director: 2587
Unknown country: 287


In [7]:
movies = df[df["type"] == "Movie"]
movies["duration_value"].describe().round(1)

count    6123.0
mean       99.6
std        28.3
min         3.0
25%        87.0
50%        98.0
75%       114.0
max       312.0
Name: duration_value, dtype: float64

In [8]:
shows = df[df["type"] == "TV Show"]
shows["duration_value"].value_counts().sort_index()

duration_value
1     1790
2      421
3      198
4       94
5       64
6       33
7       23
8       17
9        9
10       6
11       1
12       2
13       2
15       2
17       1
Name: count, dtype: int64

In [9]:
df["lag"] = df["year_added"] - df["release_year"]
df["lag"].describe().round(1)

count    8786.0
mean        4.7
std         8.8
min        -3.0
25%         0.0
50%         1.0
75%         5.0
max        93.0
Name: lag, dtype: float64

In [10]:
print("Added same year as release:", round((df["lag"] == 0).mean() * 100, 1), "%")
print("Added within 2 years:", round((df["lag"] <= 2).mean() * 100, 1), "%")
print("Added BEFORE release year:", (df["lag"] < 0).sum())

Added same year as release: 36.8 %
Added within 2 years: 63.1 %
Added BEFORE release year: 14


In [11]:
df["month_added"].value_counts()

month_added
July         827
December     812
September    769
April        763
October      758
August       754
March        739
January      737
June         728
November     705
May          632
February     562
Name: count, dtype: int64

In [12]:
type_counts = df["type"].value_counts()
type_pct = (df["type"].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({"Titles": type_counts, "Share_%": type_pct})

,Titles,Share_%
type,,
Movie,6123,69.69
TV Show,2663,30.31


In [13]:
added = df["year_added"].value_counts().sort_index().loc[2015:]
added_growth = (added.pct_change() * 100).round(1)
pd.DataFrame({"Added": added, "Growth_%": added_growth})

,Added,Growth_%
year_added,,
2015,82,NaN
2016,426,419.5
2017,1185,178.2
2018,1647,39.0
2019,2013,22.2
2020,1879,-6.7
2021,1498,-20.3


In [14]:
tv_share = (pd.crosstab(df["year_added"], df["type"], normalize="index") * 100).round(1)["TV Show"].loc[2015:]
tv_share

year_added
2015    31.7
2016    41.1
2017    29.5
2018    25.0
2019    29.4
2020    31.7
2021    33.7
Name: TV Show, dtype: float64

In [15]:
known = df[df["country"] != "Unknown"]
country_pct = (known["country"].value_counts(normalize=True) * 100).round(2)
print(country_pct.head(5))
print("Top 10 share:", country_pct.head(10).sum().round(2), "%")

country
United States     38.12
India             12.42
United Kingdom     7.51
Pakistan           4.94
Canada             3.19
Name: proportion, dtype: float64
Top 10 share: 78.02 %


In [16]:
genres = df["listed_in"].str.split(",").explode().str.strip()
(genres.value_counts().head(5) / len(df) * 100).round(1)


listed_in
International Movies      31.3
Dramas                    27.6
Comedies                  19.0
International TV Shows    15.3
Documentaries              9.9
Name: count, dtype: float64

In [17]:
(df["rating_group"].value_counts(normalize=True) * 100).round(1)

rating_group
Adults        45.6
Teens         30.1
Older Kids    13.1
Kids          10.3
Not Rated      0.9
Name: proportion, dtype: float64

In [18]:
kpis = pd.DataFrame({
    "KPI": [
        "Total titles", "Movies %", "TV Shows %", "Countries", "US share %",
        "Top 10 countries share %", "Genres", "Adults + Teens %",
        "Median movie length (min)", "TV shows with 1 season %", "Median lag (years)",
    ],
    "Value": [
        len(df),
        round((df["type"] == "Movie").mean() * 100, 1),
        round((df["type"] == "TV Show").mean() * 100, 1),
        known["country"].nunique(),
        round(country_pct.iloc[0], 1),
        round(country_pct.head(10).sum(), 1),
        genres.nunique(),
        round(df["rating_group"].isin(["Adults", "Teens"]).mean() * 100, 1),
        df[df["type"] == "Movie"]["duration_value"].median(),
        round((df[df["type"] == "TV Show"]["duration_value"] == 1).mean() * 100, 1),
        df["lag"].median(),
    ],
})
kpis

,KPI,Value
0,Total titles,8786.0
1,Movies %,69.7
2,TV Shows %,30.3
3,Countries,85.0
4,US share %,38.1
5,Top 10 countries share %,78.0
6,Genres,42.0
7,Adults + Teens %,75.7
8,Median movie length (min),98.0
9,TV shows with 1 season %,67.2


In [19]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RED, BLACK = "#E57009", "#B20303"

In [20]:
type_counts = df["type"].value_counts()
added = df.groupby(["year_added", "type"]).size().unstack(fill_value=0)
groups = ["Kids", "Older Kids", "Teens", "Adults", "Not Rated"]
rating_counts = df["rating_group"].value_counts().reindex(groups)

fig1 = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=("Content type", "Titles added per year",
                    "Audience groups", "Movies vs TV Shows added per year"),
)
fig1.add_trace(go.Pie(labels=type_counts.index, values=type_counts.values, hole=0.5,
                      marker=dict(colors=[RED, BLACK])), row=1, col=1)
fig1.add_trace(go.Scatter(x=added.index, y=added.sum(axis=1), mode="lines+markers",
                          line=dict(color=RED)), row=1, col=2)
fig1.add_trace(go.Bar(x=rating_counts.index, y=rating_counts.values,
                      marker_color=RED), row=2, col=1)
fig1.add_trace(go.Bar(x=added.index, y=added["Movie"], marker_color=RED, name="Movie"), row=2, col=2)
fig1.add_trace(go.Bar(x=added.index, y=added["TV Show"], marker_color=BLACK, name="TV Show"), row=2, col=2)
fig1.update_layout(title="Dashboard 1: Catalog Overview", height=700, barmode="stack", showlegend=False)
fig1.show()

In [21]:
known = df[df["country"] != "Unknown"]
top_c = known["country"].value_counts().head(10).sort_values()
genres = df["listed_in"].str.split(",").explode().str.strip()
top_g = genres.value_counts().head(10).sort_values()

fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Top 10 countries", "Top 10 genres"))
fig2.add_trace(go.Bar(x=top_c.values, y=top_c.index, orientation="h", marker_color=RED), row=1, col=1)
fig2.add_trace(go.Bar(x=top_g.values, y=top_g.index, orientation="h", marker_color=BLACK), row=1, col=2)
fig2.update_layout(title="Dashboard 2: Where content comes from and what it is about",
                   height=500, showlegend=False)
fig2.show()

In [22]:
movies = df[df["type"] == "Movie"]
shows = df[df["type"] == "TV Show"]
seasons = shows["duration_value"].value_counts().sort_index().loc[:6]

fig3 = make_subplots(rows=1, cols=3,
                     subplot_titles=("Movie length (minutes)", "TV show seasons",
                                     "Years from release to Netflix"))
fig3.add_trace(go.Histogram(x=movies["duration_value"], nbinsx=40, marker_color=RED), row=1, col=1)
fig3.add_trace(go.Bar(x=seasons.index, y=seasons.values, marker_color=BLACK), row=1, col=2)
fig3.add_trace(go.Histogram(x=df["lag"].clip(upper=15), nbinsx=16, marker_color=RED), row=1, col=3)
fig3.update_layout(title="Dashboard 3: Content format and freshness", height=450, showlegend=False)
fig3.show()

In [23]:
fig1.write_html("dashboard_1_overview.html")
fig2.write_html("dashboard_2_geography_genre.html")
fig3.write_html("dashboard_3_format_freshness.html")

In [24]:
shows = df[df["type"] == "TV Show"]
print(round((shows["duration_value"] >= 4).mean() * 100, 1))
print(round((df["director"] == "Unknown").mean() * 100, 1))

9.5
29.4


In [25]:
c = known["country"].value_counts()
print(round((df["lag"] <= 2).mean() * 100, 1))
print((c < 10).sum())

63.1
42


# Actionable Business Insights

*Each insight has Evidence (numbers from the data), Meaning (my interpretation) and Action (a suggestion to test, not a proven fact).*

## 1. Movies dominate, but TV Shows are gaining share
- **Evidence:** Movies are 69.7% of titles and TV Shows 30.3%. TV Shows' share of newly added titles rose from 25.0% in 2018 to 33.7% in 2021.
- **Meaning:** The catalog is shifting gradually toward series.
- **Action:** Continue growing TV content, and confirm with viewing data whether series raise engagement (this is an assumption, not shown by this dataset).

## 2. Most TV Shows have only one season
- **Evidence:** 67.2% of TV Shows (1,790 of 2,663) have 1 season, and only 9.5% have 4 or more.
- **Meaning:** Many series are short-run or still early in their run.
- **Action:** Track which one-season shows get a second season, to learn what makes a series continue.

## 3. Content comes from a few countries
- **Evidence:** The United States supplies 38.1% of titles with a known country, and the top 10 countries supply 78.0%. India is second with 12.4%. 42 of the 85 countries have fewer than 10 titles.
- **Meaning:** The catalog relies heavily on a few markets, with a long tail of small ones.
- **Action:** Keep strong coverage of the US, India and the UK, and run small tests with content from under-represented countries.

## 4. The catalog is aimed at older audiences
- **Evidence:** Adults (45.6%) and Teens (30.1%) make up 75.7% of titles. Kids and Older Kids together are 23.4%. Kids content is 17.4% of TV Shows but only 7.2% of Movies.
- **Meaning:** Children's and family content is a smaller part of the catalog, and it is served more through series than movies.
- **Action:** Consider more family and kids' movies, since this is the smallest part of the catalog.

## 5. International and drama genres lead
- **Evidence:** International Movies appear in 31.3% of titles, Dramas in 27.6% and Comedies in 19.0%. A title can carry several genres, so these shares add up to more than 100%.
- **Meaning:** Non-US stories are a core part of the catalog, in line with India and the UK ranking high by country.
- **Action:** Keep featuring international content for global audiences.

## 6. Most titles arrive soon after release
- **Evidence:** The typical (median) title reaches Netflix 1 year after release, and 63.1% arrive within 2 years. The median movie length is 98 minutes.
- **Meaning:** The catalog is mostly recent, with a smaller back-library of older titles.
- **Action:** Highlight recent releases, and group older titles as a library for classic or niche audiences.

## 7. Additions peaked in 2019 and then levelled off
- **Evidence:** Titles added per year rose to a peak of 2,013 in 2019, then fell to 1,879 in 2020 (-6.7%). The 2021 count (1,498) covers only the period up to 25 September 2021.
- **Meaning:** The fast expansion phase has slowed. The 2021 drop is mostly a data cutoff, not a confirmed decline.
- **Action:** Treat 2021 with caution, and check full-year data before concluding that additions are falling.

## Data quality and limits
- 29.4% of titles have an unknown director and 3.3% an unknown country.
- The data shows what is in the catalog, not what people watch. It has no viewing time, cost or revenue.
- Meaning and Action lines are suggestions based on patterns in the catalog. They should be tested with viewing and cost data before anyone acts on them.

## Conclusion

The data tells a consistent story: a large, recent, movie-heavy catalog aimed at older audiences, with growing emphasis on series and a strong international presence. Growth in new titles has levelled off.

## Suggested Next Steps
- Add viewing data (hours watched, completion rates) to see which content performs
- Add cost and revenue data to judge return on investment
- Rebuild these dashboards in Power BI with filters for type, country and year